# 09. 사진 보고 말하기 — VLM (openvino-genai)

> VLM(Vision Language Model)은 사진과 글을 함께 받아 글로 답한다. 여기서는 **Gemma3 4B (INT4)** 를 쓴다.
> 토큰화 · 생성 반복 · 이미지 처리는 `openvino_genai.VLMPipeline` 이 한다 — 이 단원은 **라이브러리를 잘 쓰는 법**과
> **생성 모델의 성질**(설정 · 속도 · 형식 정하기)을 배운다.

### 이 시간에 배우는 것
1. VLM 의 구조: 사진 인코더 + 연결층 + 언어 모델
2. 생성 설정: 최대 길이 · 탐욕(greedy) vs 샘플링 · 반복 억제
3. 스트리밍과 속도: **첫 글자까지 시간(TTFT)** 과 **초당 토큰**
4. 답의 **형식 정하기** (JSON) — 코드에서 쓰기 좋게
5. INT4 양자화와 메모리

### VLM 안에서 일어나는 일
```
사진 ─[사진 인코더(ViT)]─▶ 사진 토큰 수백 개 ─[연결층]─┐
질문 ─[토크나이저]─▶ 글 토큰들 ───────────────────────┴─▶ [언어 모델] ─▶ 다음 토큰 하나 ─┐
                                                             ▲                          │
                                                             └──── 붙여서 다시 ◀─────────┘  (끝 토큰이 나올 때까지 반복)
```

## 1. 모델 올리기
- **INT4**: 가중치 숫자 하나를 4비트(0.5 바이트)로 줄여 저장했다. 40억 개 × 0.5 바이트 ≈ 2GB + 사진 인코더 · 기타 → 폴더 약 3.5GB.
  정밀도를 조금 잃는 대신 메모리와 속도를 크게 아낀다.
- `CACHE_DIR` 을 주면 두 번째부터 훨씬 빨리 올라간다.
- edge-lab(`run.bat`)이 켜져 있으면 끄고 한다 (메모리 두 벌).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
size_gb = sum(f.stat().st_size for f in VLM_DIR.rglob("*") if f.is_file()) / 1e9
print(f"모델 폴더 {size_gb:.1f} GB")
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
CACHE = Path("ov-cache"); CACHE.mkdir(exist_ok=True)

t0 = time.perf_counter()
pipe = og.VLMPipeline(str(VLM_DIR), DEVICE, CACHE_DIR=str(CACHE))
print(f"올림: {time.perf_counter() - t0:.0f}초 ({DEVICE})")

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 묻고 답 받기 — 스트리밍
- 사진은 `ov.Tensor` 로 넣는다: `[1, 세로, 가로, 3]` RGB uint8. 너무 크면 사진 토큰이 늘어 느리기만 하다 — 긴 쪽 640 이 알맞다.
- `streamer` 함수를 주면 토큰이 만들어질 때마다 불린다 — 답이 한 글자씩 흘러나온다. `RUNNING` 을 돌려주면 계속, `STOP` 이면 멈춘다.

**생성 설정**
| 설정 | 뜻 |
|---|---|
| `max_new_tokens` | 최대 몇 토큰까지 만들까 (한국어 한 글자 ≈ 1~2 토큰) |
| `do_sample=False` | 탐욕: 매번 가장 확률 높은 토큰 — 같은 질문엔 같은 답 |
| `do_sample=True` + `temperature` | 샘플링: 확률대로 뽑기 — 다양하지만 들쭉날쭉. 온도가 높을수록 더 |
| `repetition_penalty` | 이미 나온 토큰의 확률을 깎아 같은 말 반복을 막는다 |

In [ ]:
def to_tensor(bgr, max_side=640):
    h, w = bgr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)))
    return ov.Tensor(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[None].astype(np.uint8))

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False
cfg.repetition_penalty = 1.15

RUNNING = og.StreamingStatus.RUNNING if hasattr(og, "StreamingStatus") else False
first = {}
def streamer(piece):
    first.setdefault("t", time.perf_counter())       # 첫 토큰이 나온 때
    print(piece, end="", flush=True)
    return RUNNING                                    # 계속 만들어라

def ask(bgr, question, quiet=False):
    first.clear()
    t0 = time.perf_counter()
    kw = {} if quiet else {"streamer": streamer}
    out = pipe.generate(question, image=to_tensor(bgr), generation_config=cfg, **kw) if bgr is not None \
        else pipe.generate(question, generation_config=cfg, **kw)
    t1 = time.perf_counter()
    n = len(pipe.get_tokenizer().encode(str(out)).input_ids.data[0])
    if not quiet:
        ttft = first.get("t", t1) - t0
        print(f"\n\n첫 글자까지 {ttft:.1f}초 · 전체 {t1 - t0:.1f}초 · 토큰 {n}개 · 약 {n / max(t1 - first.get('t', t0), 1e-6):.1f} 토큰/초")
    return str(out)

answer = ask(frame, "이 사진을 한국어로 두 문장으로 설명하세요.")

**읽는 법**
- **첫 글자까지 시간(TTFT)**: 사진 인코딩 + 질문 전체를 한 번에 읽는 시간. 사진이 클수록 길다.
- **초당 토큰**: 그 뒤 한 토큰씩 만드는 속도. 메모리 대역폭에 크게 좌우된다.

In [ ]:
answer = ask(frame, "사진 속 물건을 쉼표로 나열하세요. 설명은 쓰지 마세요.")

## 4. 답의 형식 정하기 — JSON
코드에서 답을 쓰려면 자유로운 문장보다 **정해진 형식**이 좋다. "JSON 으로만" 이라고 시키고, 받은 글에서 `{ … }` 부분만 꺼내 읽는다.
작은 모델은 가끔 형식을 어긴다 — 그래서 **실패하면 어떻게 할지** 도 코드로 정해 둔다.

In [ ]:
import json, re

def ask_json(bgr, question):
    text = ask(bgr, question + '\n반드시 JSON 하나로만 답하세요. 다른 글은 쓰지 마세요.', quiet=True)
    m = re.search(r"\{.*\}", text, re.S)                # 앞뒤 군말을 떼고 { … } 만
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

r = ask_json(frame, '사진에 대해 {"사람수": 숫자, "물건": [이름들], "분위기": "한 낱말"} 형식으로 답하세요.')
print("받은 것:", r)
if r:
    print("사람 수 + 1 =", int(r.get("사람수", 0)) + 1, "← 숫자로 바로 쓸 수 있다")

## 5. 사진 없이 글만
같은 모델에 사진을 안 주면 글 LLM 처럼 쓸 수 있다 (08 번 RAG 가 이렇게 쓴다).

In [ ]:
_ = ask(None, "OpenVINO 의 NPU 가 무엇인지 한 문장으로 설명하세요.")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 올리다가 멈춤 · 메모리 부족 | edge-lab 이 켜져 있다 / 메모리 16GB 미만 → 끄고 다시, 다른 프로그램 닫기 |
| 같은 말을 되풀이 | `repetition_penalty` 를 1.1~1.3 으로 |
| 답이 중간에 끊김 | `max_new_tokens` 가 작다 |
| JSON 이 깨짐 | 작은 모델의 한계 → 예시를 프롬프트에 넣고, 실패 처리를 꼭 둔다 |

## 정리
- VLM = 사진 인코더 + 언어 모델. 답은 토큰을 하나씩 이어 붙여 만든다.
- 속도는 TTFT(사진 · 질문 읽기)와 초당 토큰(만들기)으로 나눠 본다.
- 코드에 쓸 답은 형식(JSON)을 정해 받고, 어겼을 때의 처리도 만든다.

---
# 🛠 미니 프로젝트 — 사진 일기 생성기

**만들 것**: 웹캠으로 사진을 몇 장 찍으면(또는 `photos/` 폴더의 사진), 장마다 VLM 이 **제목 · 설명 · 태그**를 JSON 으로 만들고,
사진과 함께 **마크다운 일기 파일**(`diary.md`)로 저장한다.

**쓰는 것**: `to_tensor` · `ask_json` (이 노트북), 사진 저장(`cv2.imwrite`)

**순서**
1. 사진 모으기 — `photos/*.jpg` 가 있으면 그것, 없으면 웹캠으로 3장(5초 간격)
2. 장마다 `ask_json` → `{"제목", "설명", "태그"}`
3. `diary.md` 에 날짜 · 사진 · 내용을 쓴다

In [ ]:
from datetime import datetime

shots = [cv2.imread(str(p)) for p in sorted(Path("photos").glob("*.jpg"))] if Path("photos").is_dir() else []
if not shots:
    for i in range(3):
        print(f"{i + 1}/3 찍는 중 ..."); shots.append(grab()); time.sleep(5)

Path("diary").mkdir(exist_ok=True)
lines = [f"# 사진 일기 — {datetime.now():%Y-%m-%d}", ""]
for i, img in enumerate(shots):
    t0 = time.perf_counter()
    r = ask_json(img, '이 사진으로 일기를 쓰려고 합니다. {"제목": "짧게", "설명": "두 문장", "태그": ["낱말", "낱말"]} 형식으로 답하세요.') \
        or {"제목": "(못 만듦)", "설명": "", "태그": []}
    name = f"diary/{datetime.now():%H%M%S}_{i}.jpg"
    cv2.imwrite(name, img)
    lines += [f"## {r.get('제목', '')}", f"![]({name})", "", str(r.get("설명", "")), "",
              " ".join(f"#{t}" for t in r.get("태그", [])), ""]
    print(f"{i + 1}. {r.get('제목')}  ({time.perf_counter() - t0:.1f}초)")

Path("diary.md").write_text("\n".join(lines), encoding="utf-8")
print("저장: diary.md — 왼쪽 파일 목록에서 열면 사진과 함께 보인다")

**더 해 보기**
- 태그별로 사진을 모아 보여 주는 "태그 목록" 을 일기 끝에 붙이기
- 사진 여러 장을 한 번에 보고 "오늘 하루 요약" 한 문단 만들기 (장마다 받은 설명을 글로 모아 `ask(None, ...)`)